# Verifying Symbolic-Rotation Circuits with `PhasedOutcomeCompleteSimulation`

This notebook shows how to apply a **symbolic rotation** $e^{i\alpha P}$ to a
`PhasedOutcomeCompleteSimulation` and use it to verify equivalence of non-stabilizer circuits, the
motivating application of [arXiv:2603.24717](https://arxiv.org/abs/2603.24717). (See the companion
notebook *Tracking the Exact Global Phase* for an introduction to the phased simulator itself.)

## Modeling a symbolic rotation

An arbitrary-angle rotation $e^{i\alpha P} = \cos(\alpha)\,I + i\sin(\alpha)\,P$ is **not** a
stabilizer operation, so it cannot be applied directly. (Note that `apply_pauli_exp` is only the
*fixed* Clifford rotation $e^{i\pi/4\,P}$, not a free-angle rotation.)

Following §4.1 of the paper, we instead apply the Pauli $P$ **conditioned on a fresh random bit**
$a$. The outcome-complete machinery then tracks both branches at once:

- $a = 0$: the identity branch, carrying amplitude weight $\cos\alpha$,
- $a = 1$: the $P$ branch, carrying amplitude weight $i\sin\alpha$,

so that $e^{i\alpha P}|\psi\rangle = \cos(\alpha)\,|\text{branch }0\rangle + i\sin(\alpha)\,|\text{branch }1\rangle$
for **every** $\alpha$. Because the phased simulator keeps the *exact* global phase of each branch,
two circuits agree for all $\alpha$ iff their branch states match exactly — phase included.

Nothing here forms a $2^n$ state vector: phases are exact integer powers of $\zeta_8 = e^{i\pi/4}$ and
every step is polynomial in the number of qubits.

## Setup

In [1]:
from paulimer import (
    PhasedOutcomeCompleteSimulation,
    PhasedCircuitAction,
    SparsePauli,
    UnitaryOpcode,
)

ZETA8_LABEL = {0: "1", 1: "ζ₈", 2: "i", 3: "ζ₈³", 4: "-1", 5: "ζ₈⁵", 6: "-i", 7: "ζ₈⁷"}

## A single symbolic rotation, two branches

We build $C_1\, e^{i\alpha Z_0}\, C_2\,|0\rangle$ with $C_2 = H_0$ and $C_1 = H_0$, modeling the
rotation by a `Z_0` conditioned on a fresh random bit.

In [2]:
sim = PhasedOutcomeCompleteSimulation(1)
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])              # C_2
a = sim.allocate_symbolic_angle()                           # the rotation's symbolic angle
sim.apply_conditional_pauli(SparsePauli("Z_0"), [a])        # e^{iα Z_0}  ->  Z_0^a
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])              # C_1

print("random outcomes:", sim.random_outcome_count)
for value in (0, 1):
    e = sim.output_phase_exponent([bool(value)])
    print(f"  branch a={value}:  zeta8^{e} = {ZETA8_LABEL[e]}"
          f"   (amplitude weight {'cos α' if value == 0 else 'i·sin α'})")

random outcomes: 1
  branch a=0:  zeta8^0 = 1   (amplitude weight cos α)
  branch a=1:  zeta8^0 = 1   (amplitude weight i·sin α)


## Verifying equivalence over all inputs with Choi states

To check that two circuits implement the **same operator** on an *unknown* input, it is not enough to
run them on one fixed state — we must compare them on every input at once. Channel–state duality lets
us do this with a single stabilizer state: entangle each of the $n$ system qubits with a fresh
*reference* qubit via a Bell pair $|\Phi\rangle$, then apply the circuit to the system qubits only.
The resulting **Choi state** $(U \otimes I)\,|\Phi\rangle^{\otimes n}$ determines $U$ completely.

The phased simulator packages exactly this comparison: build the Choi state in the simulator, then call
`sim.phased_action(input_qubits, output_qubits)` to obtain a `PhasedCircuitAction`. Two actions are
then compared directly:

- `a.is_equivalent(b)` — equal as operators on every input, **including** the exact relative phases
  between branches (up to one overall global phase);
- `a.is_equivalent_up_to_signs(b)` — equal only in their stabilizer (symplectic) action, ignoring all
  phases — i.e. precisely what an ordinary, phase-blind stabilizer simulation sees.

Each symbolic rotation angle is one `allocate_random_bit()`, and the comparison matches these angle
bits **one-to-one** between the two circuits.

In [3]:
def choi_action(build_gadget, n=1):
    """Phased Choi action of a gadget: Bell-pair each system qubit q in 0..n with its reference q+n,
    allocate one symbolic angle, then apply the gadget to the system qubits only."""
    sim = PhasedOutcomeCompleteSimulation(2 * n)
    for q in range(n):
        sim.apply_unitary(UnitaryOpcode.PrepareBell, [q, q + n])
    angle = sim.allocate_symbolic_angle()
    build_gadget(sim, angle)
    return sim.phased_action(list(range(n)), list(range(n)))


def conjugated_z(sim, a):        # H · e^{iα Z0} · H
    sim.apply_unitary(UnitaryOpcode.Hadamard, [0])
    sim.apply_conditional_pauli(SparsePauli("Z_0"), [a])
    sim.apply_unitary(UnitaryOpcode.Hadamard, [0])

def x_rotation(sim, a):          # e^{iα X0}
    sim.apply_conditional_pauli(SparsePauli("X_0"), [a])

assert choi_action(conjugated_z).is_equivalent(choi_action(x_rotation))
print("✓ H · e^{iα Z} · H  ==  e^{iα X}   (verified as operators over all inputs)")

✓ H · e^{iα Z} · H  ==  e^{iα X}   (verified as operators over all inputs)


## A multi-qubit, entangling equivalence

The idiom extends verbatim to **entangling** rotations of arbitrary Pauli weight — no new machinery.
A clean example: conjugating a single-qubit rotation by a `CNOT` turns it into a two-qubit
$ZZ$ rotation,
$$\mathrm{CNOT}_{01}\; e^{i\alpha Z_1}\; \mathrm{CNOT}_{01} \;=\; e^{i\alpha Z_0 Z_1},$$
because $\mathrm{CNOT}_{01}$ conjugates $Z_1 \mapsto Z_0 Z_1$. We verify it as operators (Choi state,
$n=2$), and confirm that dropping the conjugation — a bare $e^{i\alpha Z_1}$ — is genuinely different.

In [4]:
def zz_direct(sim, a):           # e^{iα Z0 Z1}
    sim.apply_conditional_pauli(SparsePauli("Z_0 Z_1"), [a])

def zz_via_cnot(sim, a):         # CNOT01 · e^{iα Z1} · CNOT01
    sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])
    sim.apply_conditional_pauli(SparsePauli("Z_1"), [a])
    sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])

def z1_only(sim, a):             # e^{iα Z1}   (conjugation dropped)
    sim.apply_conditional_pauli(SparsePauli("Z_1"), [a])

assert choi_action(zz_direct, n=2).is_equivalent(choi_action(zz_via_cnot, n=2))
print("✓ e^{iα Z0Z1}  ==  CNOT01 · e^{iα Z1} · CNOT01   (entangling, verified as operators)")

assert not choi_action(zz_direct, n=2).is_equivalent(choi_action(z1_only, n=2))
print("✓ e^{iα Z0Z1}  !=  e^{iα Z1}                     (the CNOT conjugation genuinely matters)")

✓ e^{iα Z0Z1}  ==  CNOT01 · e^{iα Z1} · CNOT01   (entangling, verified as operators)
✓ e^{iα Z0Z1}  !=  e^{iα Z1}                     (the CNOT conjugation genuinely matters)


## A phase difference that ordinary simulation misses

Finally, a difference that is *purely* a phase. The rotations $e^{+i\alpha Z}$ and $e^{-i\alpha Z}$
condition $+Z$ and $-Z$ on the angle bit. Since $+Z$ and $-Z$ have the **same symplectic action**
(they differ only by a sign), an ordinary stabilizer simulation cannot tell them apart — *even* under
the Choi comparison above. Yet they are physically different, differing by a relative $-1$ on the
$a = 1$ branch. The two `PhasedCircuitAction` comparisons make this explicit: `is_equivalent_up_to_signs`
(phase-blind) reports them equal, while `is_equivalent` (phase-aware) separates them.

In [5]:
def rot_pos(sim, a):             # e^{+iα Z0}   ->  +Z0 on the a=1 branch
    sim.apply_conditional_pauli(SparsePauli("Z_0"), [a])

def rot_neg(sim, a):             # e^{-iα Z0}   ->  -Z0 on the a=1 branch
    sim.apply_conditional_pauli(SparsePauli("-Z_0"), [a])

pos, neg = choi_action(rot_pos), choi_action(rot_neg)

assert pos.is_equivalent_up_to_signs(neg)
print("Ignoring phase (is_equivalent_up_to_signs):  e^{+iα Z} and e^{-iα Z} are INDISTINGUISHABLE")

assert not pos.is_equivalent(neg)
print("Tracking phase (is_equivalent):              e^{+iα Z} != e^{-iα Z}")

Ignoring phase (is_equivalent_up_to_signs):  e^{+iα Z} and e^{-iα Z} are INDISTINGUISHABLE
Tracking phase (is_equivalent):              e^{+iα Z} != e^{-iα Z}


In [6]:
def branch_phases(build_gadget):
    """The exact ζ₈ exponent on each branch of the gadget's Choi state."""
    sim = PhasedOutcomeCompleteSimulation(2)
    sim.apply_unitary(UnitaryOpcode.PrepareBell, [0, 1])
    a = sim.allocate_symbolic_angle()
    build_gadget(sim, a)
    return tuple(sim.output_phase_exponent([bool(v)]) for v in (0, 1))

for name, gadget in (("e^{+iα Z}", rot_pos), ("e^{-iα Z}", rot_neg)):
    phases = branch_phases(gadget)
    labels = tuple(ZETA8_LABEL[e] for e in phases)
    print(f"{name} branch phases: exponents {phases}  =  {labels}")
print("\nThe a=1 branch differs by ζ₈⁴ = -1 — exactly the relative phase ordinary simulation discards.")

e^{+iα Z} branch phases: exponents (0, 0)  =  ('1', '1')
e^{-iα Z} branch phases: exponents (0, 4)  =  ('1', '-1')

The a=1 branch differs by ζ₈⁴ = -1 — exactly the relative phase ordinary simulation discards.


## Mixing virtual and true bits: "ejection"

A symbolic rotation can be executed **remotely**. Copy the system qubit onto a fresh ancilla with
a `CNOT`, apply the symbolic $Z$-rotation to the **ancilla**, then measure the ancilla in the $X$
basis; a $-$ outcome triggers a conditional $Z$ correction on the system qubit. This *ejection*
gadget mixes a **virtual** angle bit (the rotation, via `allocate_symbolic_angle()`) with a **true**
measurement bit (the $X$ read-out, allocated internally by `measure`). Tracing out the true bit, the
channel it implements is exactly the direct rotation $e^{i\alpha Z}$ on the input.

In [7]:
def eject_z_rotation(sim, system, ancilla, angle):
    """Remotely execute e^{iα Z_system} using an ancilla, an X measurement, and a Z correction."""
    sim.apply_unitary(UnitaryOpcode.ControlledX, [system, ancilla])      # copy system -> ancilla
    sim.apply_conditional_pauli(SparsePauli(f"Z_{ancilla}"), [angle])    # symbolic rotation on ancilla
    outcome = sim.measure(SparsePauli(f"X_{ancilla}"))                   # true measurement bit
    sim.apply_conditional_pauli(SparsePauli(f"Z_{system}"), [outcome])   # conditional Z correction

# Direct:  e^{iα Z_0} on the system qubit (qubit 0, reference qubit 1).
direct = PhasedOutcomeCompleteSimulation(2)
direct.apply_unitary(UnitaryOpcode.PrepareBell, [0, 1])
a = direct.allocate_symbolic_angle()
direct.apply_conditional_pauli(SparsePauli("Z_0"), [a])
direct_action = direct.phased_action([0], [0])

# Ejected: system qubit 0 (reference qubit 1), ancilla qubit 2.
ejected = PhasedOutcomeCompleteSimulation(3)
ejected.apply_unitary(UnitaryOpcode.PrepareBell, [0, 1])
a = ejected.allocate_symbolic_angle()        # virtual: the rotation angle
eject_z_rotation(ejected, 0, 2, a)           # ... and one true measurement bit inside
ejected_action = ejected.phased_action([0], [0])

assert direct_action.is_equivalent(ejected_action)
print("✓ remotely ejected e^{iα Z} == direct e^{iα Z}  (virtual angle + true measurement bit)")

✓ remotely ejected e^{iα Z} == direct e^{iα Z}  (virtual angle + true measurement bit)


## Summary

- A symbolic rotation $e^{i\alpha P}$ is applied by conditioning the Pauli $P$ on a fresh
  `allocate_symbolic_angle()` and calling `apply_conditional_pauli(P, [a])`. This works for an
  **arbitrary** Pauli $P$ of any weight — multi-qubit and entangling rotations need nothing new.
- To compare two circuits as **operators** on an unknown input, build their **Choi states**
  (Bell-pair every system qubit with a reference qubit, apply the circuit to the system qubits) and
  call `sim.phased_action(...)`. The resulting `PhasedCircuitAction` objects compare with
  `is_equivalent` (phase-aware) and `is_equivalent_up_to_signs` (phase-blind).
- The phased simulator tracks both branches with their **exact** $\zeta_8$ phase — precisely the
  information ordinary stabilizer simulation throws away. Here it is the only thing distinguishing
  $e^{+i\alpha Z}$ from $e^{-i\alpha Z}$, whose conditioned Paulis $+Z$ and $-Z$ share a symplectic
  action.

> **Note.** `allocate_symbolic_angle()` tags a *virtual* angle bit, distinct from a *true*
> measurement bit (`allocate_random_bit()`, or the bit produced by `measure`). `is_equivalent`
> matches virtual angles one-to-one across the two circuits while marginalizing true measurement
> bits — so the ejection gadget below, which mixes both kinds, is recognized as equivalent to the
> direct rotation.